In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta, UTC

from sgp4.api import Satrec, jday

from skyfield.api import EarthSatellite, load


pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
from skyfield.api import load, EarthSatellite
from scipy.spatial import KDTree
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from datetime import datetime, timedelta, UTC

ts = load.timescale()

# 🛰️ Orbital Data Analysis Dashboard
### Project: Space Sustainability & Debris Tracking

This notebook processes data from **Space-Track.org** to analyze the behavior of active satellites and space debris.

---

## 📋 Data Dictionary (GP Class)

### 1. Identity Information (الهوية)
* **`OBJECT_NAME`**: Common name (e.g., TIROS 1).
* **`NORAD_CAT_ID`**: Unique ID (e.g., 29).
* **`OBJECT_TYPE`**: 
    * `PAYLOAD`: Active/Inactive Satellite.
    * `DEBRIS`: Space Junk.
* **`COUNTRY_CODE`**: Owning nation (e.g., US, SA, etc.).

### 2. Orbital Elements (العناصر المدارية)

* **`INCLINATION`**: Orbit angle relative to the equator (Degrees).
* **`ECCENTRICITY`**: How elliptical the orbit is (0 = circle).
* **`MEAN_MOTION`**: Orbits per day (Speed).
* **`PERIOD`**: Time for one orbit (Minutes).
* **`APOAPSIS` / `PERIAPSIS`**: Highest and lowest points (Altitude in km).

### 3. Physics & Re-entry (الحالة والفيزياء)
* **`BSTAR`**: Drag coefficient (How much the atmosphere slows it down).
* **`RCS_SIZE`**: Size based on radar (Small, Medium, Large).
* **`DECAY_DATE`**: Date of re-entry (If already burned up).

In [ ]:
df = pd.read_csv('orbital_data_export_1.csv')
df.head()

In [ ]:
type_counts = df['OBJECT_TYPE'].value_counts()

In [ ]:
plt.figure(figsize=(10,6))

type_counts.plot(kind='bar', color=['#e74c3c', '#3498db', '#2ecc71'])
plt.title('d')
plt.xlabel('obj type')
plt.ylabel('total count')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


In [ ]:
df['OBJECT_TYPE'][0]

In [ ]:
now = datetime.now(UTC)
jd, fr = jday(now.year, now.month, now.day, now.hour, now.minute, now.second)

ts = load.timescale()
t_sf = ts.utc(now.year, now.month, now.day, now.hour, now.minute, now.second)

In [ ]:
results = []

for _, row in df.iterrows():
    try:
        line1 = row['TLE_LINE1']
        line2 = row['TLE_LINE2']

        # --- SGP4 (ECI)
        sat_sgp4 = Satrec.twoline2rv(line1, line2)
        e, r, v = sat_sgp4.sgp4(jd, fr)

        if e != 0:
            continue

        # --- Skyfield (Lat/Lon)
        sat_sf = EarthSatellite(line1, line2)
        geocentric = sat_sf.at(t_sf)
        subpoint = geocentric.subpoint()

        lat = subpoint.latitude.degrees
        lon = subpoint.longitude.degrees
        alt = subpoint.elevation.km

        results.append({
            "name": row['OBJECT_NAME'],
            "type": row['OBJECT_TYPE'],
            "lat": lat,
            "lon": lon,
            "alt": alt
        })

    except Exception as ex:
        continue

df_vis = pd.DataFrame(results)

print(df_vis.head())
print("Total objects:", len(df_vis))

In [ ]:
color_map = {
    "PAYLOAD": "green",
    "DEBRIS": "red",
    "ROCKET BODY": "orange"
}

df_vis["color"] = df_vis["type"].map(color_map).fillna("white")


In [ ]:
import plotly.graph_objects as go
import plotly.io as pio

# إعداد المخرج لـ Jupyter
pio.renderers.default = "notebook_connected"

fig = go.Figure()

fig.add_trace(go.Scattergeo(
    lat=df_vis["lat"],
    lon=df_vis["lon"],
    mode="markers",
    text=df_vis["name"],
    marker=dict(
        size=6,
        color=df_vis["color"],
        opacity=0.7
    )
))

fig.update_layout(
    geo=dict(
        projection_type='orthographic',
        showland=True,
        showocean=True,
        bgcolor="black"
    ),
    title="Space Objects Tracking (Satellites & Debris)",
    margin={"r":0,"t":40,"l":0,"b":0}
)

fig.show()

In [ ]:
# ts = load.timescale()
# start_time = datetime.now(UTC)

# # عدد الفريمات (زمن الحركة)
# num_frames = 20
# step_minutes = 2

# frames = []

# # ألوان حسب النوع
# color_map = {
#     "PAYLOAD": "green",
#     "DEBRIS": "red",
#     "ROCKET BODY": "orange"
# }

# # نحدد عدد أجسام أقل بالبداية (لتجنب البطء)
# df_sample = df.head(200)  # جرب 100-300 بالبداية

# for i in range(num_frames):
#     current_time = start_time + timedelta(minutes=i * step_minutes)

#     t = ts.utc(
#         current_time.year, current_time.month, current_time.day,
#         current_time.hour, current_time.minute, current_time.second
#     )

#     lats, lons, colors, texts = [], [], [], []

#     for _, row in df_sample.iterrows():
#         try:
#             sat = EarthSatellite(row['TLE_LINE1'], row['TLE_LINE2'])
#             subpoint = sat.at(t).subpoint()

#             lats.append(subpoint.latitude.degrees)
#             lons.append(subpoint.longitude.degrees)
#             colors.append(color_map.get(row['OBJECT_TYPE'], "white"))
#             texts.append(row['OBJECT_NAME'])

#         except:
#             continue

#     frame = go.Frame(
#         data=[go.Scattergeo(
#             lat=lats,
#             lon=lons,
#             mode="markers",
#             text=texts,
#             marker=dict(size=4, color=colors)
#         )],
#         name=str(i)
#     )

#     frames.append(frame)

# # الشكل الأساسي
# fig = go.Figure(
#     data=frames[0].data,
#     frames=frames
# )

# # أزرار التشغيل
# fig.update_layout(
#     geo=dict(
#         projection_type='orthographic',
#         showland=True,
#         showocean=True,
#         bgcolor="black"
#     ),
#     updatemenus=[{
#         "type": "buttons",
#         "buttons": [
#             {
#                 "label": "Play",
#                 "method": "animate",
#                 "args": [None, {"frame": {"duration": 500, "redraw": True}}]
#             }
#         ]
#     }],
#     title="Satellite & Debris Motion (Animation)"
# )

# fig.show()

In [ ]:
def compute_positions(df, ts, time):

    results = []

    t = ts.utc(
        time.year,
        time.month,
        time.day,
        time.hour,
        time.minute,
        time.second
    )

    for _, row in df.iterrows():
        try:
            sat = EarthSatellite(row['TLE_LINE1'], row['TLE_LINE2'])

            geocentric = sat.at(t)
            subpoint = geocentric.subpoint()

            pos = geocentric.position.km
            vel = geocentric.velocity.km_per_s

            results.append({
                "name": row["OBJECT_NAME"],
                "type": row["OBJECT_TYPE"],

                # visualization
                "lat": subpoint.latitude.degrees,
                "lon": subpoint.longitude.degrees,

                # physics
                "x": pos[0],
                "y": pos[1],
                "z": pos[2],

                "vx": vel[0],
                "vy": vel[1],
                "vz": vel[2]
            })

        except:
            continue

    return pd.DataFrame(results)

In [ ]:
def calculate_risk(obj1, obj2):

    pos1 = np.array([obj1['x'], obj1['y'], obj1['z']])
    pos2 = np.array([obj2['x'], obj2['y'], obj2['z']])

    vel1 = np.array([obj1['vx'], obj1['vy'], obj1['vz']])
    vel2 = np.array([obj2['vx'], obj2['vy'], obj2['vz']])

    distance = np.linalg.norm(pos1 - pos2)
    rel_vel = np.linalg.norm(vel1 - vel2)

    if distance < 1:
        return "HIGH"
    elif distance < 2:
        return "MEDIUM"
    else:
        return "LOW"

In [ ]:
def detect_collisions(df_positions, threshold=2):

    df_positions = df_positions.dropna(subset=['x','y','z'])

    positions = df_positions[['x','y','z']].values

    if len(positions) == 0:
        return []

    tree = KDTree(positions)
    pairs = tree.query_pairs(r=threshold)

    collisions = []

    for i, j in pairs:

        obj1 = df_positions.iloc[i]
        obj2 = df_positions.iloc[j]

        if (
            (obj1['type'] == 'DEBRIS' and obj2['type'] == 'PAYLOAD') or
            (obj2['type'] == 'DEBRIS' and obj1['type'] == 'PAYLOAD')
        ):
            collisions.append({
                "obj1": obj1['name'],
                "obj2": obj2['name'],
                "risk": calculate_risk(obj1, obj2)
            })

    return collisions

In [ ]:
active_events = {}

start_time = datetime.now(UTC)

for i in range(20):

    current_time = start_time + timedelta(minutes=i)

    df_positions = compute_positions(df, ts, current_time)

    collisions = detect_collisions(df_positions)

    current_pairs = set()

    for c in collisions:

        key = (c['obj1'], c['obj2'])
        current_pairs.add(key)

        if key not in active_events:
            active_events[key] = {
                "start": current_time,
                "end": current_time,
                "risk": c['risk']
            }
        else:
            active_events[key]["end"] = current_time
            active_events[key]["risk"] = c['risk']

In [ ]:
frames = []

start_time = datetime.now(UTC)

for i in range(20):

    current_time = start_time + timedelta(minutes=i)

    df_positions = compute_positions(df, ts, current_time)

    collisions = detect_collisions(df_positions)

    risky = set()
    for c in collisions:
        risky.add(c['obj1'])
        risky.add(c['obj2'])

    colors = []

    for _, row in df_positions.iterrows():
        if row['name'] in risky:
            colors.append("yellow")
        else:
            colors.append({
                "PAYLOAD": "green",
                "DEBRIS": "red",
                "ROCKET BODY": "orange"
            }.get(row['type'], "white"))

    frame = go.Frame(
        data=[go.Scattergeo(
            lat=df_positions["lat"],
            lon=df_positions["lon"],
            mode="markers",
            marker=dict(size=4, color=colors, opacity=0.7),
            text=df_positions["name"]
        )],
        name=str(i)
    )

    frames.append(frame)

In [ ]:
fig = go.Figure(
    data=frames[0].data,
    frames=frames
)

In [ ]:
fig.update_layout(
    geo=dict(
        projection_type="orthographic",
        showland=True,
        showocean=True,
        landcolor="rgb(20,100,20)",
        oceancolor="rgb(10,20,50)"
    ),
    title="Space Debris Simulation (Moving System)",
    updatemenus=[{
        "type": "buttons",
        "buttons": [{
            "label": "Play",
            "method": "animate",
            "args": [None, {
                "frame": {"duration": 500, "redraw": True},
                "fromcurrent": True
            }]
        }]
    }]
)

fig.show()

In [ ]:
import numpy as np

def predict_collision_risk(df, ts, start_time, days=30, step_hours=6):

    risks = []

    steps = int((days * 24) / step_hours)

    for i in range(steps):

        future_time = start_time + timedelta(hours=i * step_hours)

        df_future = compute_positions(df, ts, future_time)

        for _, a in df_future.iterrows():
            for _, b in df_future.iterrows():

                if a['name'] == b['name']:
                    continue

                # Debris vs Payload only
                if not (
                    (a['type'] == 'DEBRIS' and b['type'] == 'PAYLOAD') or
                    (b['type'] == 'DEBRIS' and a['type'] == 'PAYLOAD')
                ):
                    continue

                pos_a = np.array([a['x'], a['y'], a['z']])
                pos_b = np.array([b['x'], b['y'], b['z']])

                distance = np.linalg.norm(pos_a - pos_b)

                if distance < 2:
                    risks.append({
                        "time": future_time,
                        "obj1": a['name'],
                        "obj2": b['name'],
                        "risk_score": 1 / (distance + 0.1)
                    })

    return risks

In [ ]:
def generate_orbit_trails(df, ts, start_time, steps=50):

    trails = {}

    for _, row in df.iterrows():
        trails[row['OBJECT_NAME']] = {"lat": [], "lon": []}

    for i in range(steps):

        t = start_time + timedelta(minutes=i)

        df_pos = compute_positions(df, ts, t)

        for _, row in df_pos.iterrows():
            name = row['name']

            trails[name]["lat"].append(row["lat"])
            trails[name]["lon"].append(row["lon"])

    return trails

In [ ]:
def plot_trails(trails, df_positions):

    fig = go.Figure()

    # trails
    for name, data in trails.items():
        fig.add_trace(go.Scattergeo(
            lat=data["lat"],
            lon=data["lon"],
            mode="lines",
            line=dict(width=1),
            opacity=0.5,
            name=name
        ))

    # current positions
    fig.add_trace(go.Scattergeo(
        lat=df_positions["lat"],
        lon=df_positions["lon"],
        mode="markers",
        marker=dict(size=4, color="red"),
        name="Current"
    ))

    fig.update_layout(
        geo=dict(
            projection_type="orthographic",
            showland=True,
            showocean=True,
            landcolor="rgb(20,100,20)",
            oceancolor="rgb(10,20,50)"
        ),
        title="Orbital Trails + Satellites"
    )

    fig.show()

In [ ]:
start_time = datetime.now(UTC)

# 1) trails
trails = generate_orbit_trails(df, ts, start_time)

# 2) current state
df_positions = compute_positions(df, ts, start_time)

# 3) plot
plot_trails(trails, df_positions)

# 4) AI prediction
risks = predict_collision_risk(df, ts, start_time)

print("⚠️ Future collision risks (30 days):")
for r in risks[:10]:
    print(r)